In [4]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [5]:
# find . -type f | sort

In [12]:
study_files = {
    "Chen2021": {
        "matrix": "Data_Chen2021_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Chen2021_Prostate/Genes.txt",
        "cells":  "Data_Chen2021_Prostate/Cells.csv",
        "samples": "Data_Chen2021_Prostate/Samples.csv",
        "units": "UMI"
    },
    "Dong2020": {
        "matrix": "Data_Dong2020_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Dong2020_Prostate/Genes.txt",
        "cells":  "Data_Dong2020_Prostate/Cells.csv",
        "samples": "Data_Dong2020_Prostate/Samples.csv",
        "units": "UMI"
    },
    "He2021": {
        "matrix": "Data_He2021_Prostate/Exp_data_TPM.mtx",
        "genes":  "Data_He2021_Prostate/Genes.txt",
        "cells":  "Data_He2021_Prostate/Cells.csv",
        "samples": "Data_He2021_Prostate/Samples.csv",
        "units": "TPM"
    },
    "Song2022": {
        "matrix": "Data_Song2022_Prostate/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Song2022_Prostate/Genes.txt",
        "cells":  "Data_Song2022_Prostate/Cells.csv",
        "samples": "Data_Song2022_Prostate/Samples.csv",
        "units": "UMI"
    }
}

In [13]:
study_name = 'Chen2021'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Prostate'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 36,424 cells × 25,044 genes
   cell_type: 5997 NaN
   cell_type: 5997 suspicious
Flagged 5997 cells
After metadata drop: 30,427 cells
   Added cancer_type column. Unique values: ['Prostate Cancer']
After gene filter: 30,427 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 16.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.9%
Elapsed time: 34.9 seconds
After doublet removal: 30,385 cells
After MT filter: 30,385 cells
Normalised (UMI) – max value 13.16


In [14]:
study_name = 'Dong2020'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Prostate'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 21,292 cells × 15,709 genes
   cell_type: 6316 NaN
   cell_type: 6316 suspicious
Flagged 6316 cells
After metadata drop: 14,976 cells
   Added cancer_type column. Unique values: ['Prostate Cancer']
After gene filter: 14,976 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 35.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 21.3 seconds
After doublet removal: 14,957 cells
After MT filter: 14,888 cells
Normalised (UMI) – max value 13.54


In [15]:
study_name = 'He2021'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Prostate'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 2,170 cells × 45,895 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Prostate Cancer']
After gene filter: 2,151 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.42
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 43.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 3.4 seconds
After doublet removal: 2,151 cells
After MT filter: 2,151 cells
Normalised (TPM) – max value 12.98


In [16]:
study_name = 'Song2022'
paths = study_files[study_name]

# matrix
adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()

cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# drop NaN
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# type col should exist
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        adata.obs['cancer_type'] = adata.obs['sample'].map(samples['cancer_type'])
    else:
        adata.obs['cancer_type'] = 'Prostate'   # PER STUDY CHANGE
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# gene filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# mitochondria filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# normalization
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

# convert object cols and save
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 21,743 cells × 21,877 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Prostate Cancer']
After gene filter: 21,540 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.72
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 3.4%
Elapsed time: 19.8 seconds
After doublet removal: 21,536 cells
After MT filter: 20,355 cells
Normalised (UMI) – max value 12.87


In [17]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Chen2021  –  30,385 cells, 25,044 genes
 cancer_type: Prostate Cancer
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_type: B_cell, Endothelial, Epithelial, Fibroblast, Macrophage, Malignant, Mast, T_cell
 mp_assignment: 55 unique values – first 15: Androgen-prostate, CAF1, CAF5, CAF9, CD4 - Cytotoxic, CD4 - Dysfunction, CD4 - Stress/HSP, CD4 - Treg, CD8 - Dysfunction, CD8 - Heat shock, CD8 - Memory/Naive1, CD8 - Naive2, Cell Cycle, Cell Cycle - G1/S, Cell Cycle - G2/M
 mp_top: 127 unique values – first 15: Alveolar, Androgen-prostate, Astrocytes, B-cells1, CAF1, CAF10, CAF2, CAF3, CAF4, CAF5, CAF6, CAF7, CAF8, CAF9, CD4 - Cell Cycle
 study: Chen2021

Dong2020  –  14,888 cells, 15,709 genes
 cancer_type: Prostate Cancer
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling
 cell_type: Endothelial, Epithelial, Fibroblast, Lymphoid, Malignant, Myeloid
 mp_assignment: 38 unique values – first 15: Androgen-prostate, CAF1, CAF5, CAF8, Cell Cycle, Cell Cycle - G1/S, Ce

In [19]:

INPUT_PATTERN = "*.h5ad"   # finds all normalized study files
OUTPUT_FILE = "prostate.h5ad"

valid_prefixes = ['Chen2021', 'Dong2020', 'He2021', 'Song2022']
all_files = sorted(glob.glob(INPUT_PATTERN))
real_files = [f for f in all_files if any(f.startswith(p) for p in valid_prefixes)]
print(f"Found {len(real_files)} prostate study files (ignored {len(all_files)-len(real_files)} other files)\n")

def label_malignant(adata, study_name):

    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    
    #  disease column (e.g., He2021 has 'metastatic prostate carcinoma')
    if 'disease' in adata.obs.columns:
        disease = adata.obs['disease'].astype(str).str.lower().str.strip()
        disease_kw = ['cancer', 'tumor', 'malignant', 'carcinoma', 'prostate', 'metastatic']
        mask = disease.str.contains('|'.join(disease_kw), na=False)
        if mask.any():
            return mask, f"disease column: {disease[mask].unique()[:3]}"
    
    # source column + epithelial (just in case)
    if 'source' in adata.obs.columns and 'cell_type' in adata.obs.columns:
        src = adata.obs['source'].astype(str).str.lower().str.strip()
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        is_target = ct.str.contains('epithelial', na=False)
        is_tumor_source = src.str.contains('tumor', na=False)
        mask = is_target & is_tumor_source
        if mask.any():
            return mask, "source='Tumor' & epithelial"
    
    # If nothing matched, all non‑malignant
    return pd.Series(False, index=adata.obs.index), "all non‑malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in real_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]
    
    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)
    
    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    
    print(f"{study_name:<15s}  {adata.n_obs:>7,} cells  ->  malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")
    
    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  ->  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")


adatas = [sc.read_h5ad(tf) for tf in temp_files]
adata_all = adatas[0].concatenate(
    adatas[1:],
    batch_key='study',
    join='inner',
    index_unique='-'
)
print(f"Merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")


essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
keep = [c for c in essential if c in adata_all.obs.columns]
adata_all.obs = adata_all.obs[keep]
print(f"Metadata retained: {list(adata_all.obs.columns)}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

for col in keep:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

print(f"Final clean dataset: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")
for col in adata_all.obs.columns:
    if adata_all.obs[col].dtype == 'object':
        adata_all.obs[col] = adata_all.obs[col].astype(str)
adata_all.write_h5ad(OUTPUT_FILE)

for tf in temp_files:
    os.remove(tf)
print("Temporary files cleaned up.")

Found 4 prostate study files (ignored 4 other files)

Chen2021          30,385 cells  ->  malignant: 14,285   non‑malignant:  16,100   (cell_type='Malignant')
Dong2020          14,888 cells  ->  malignant:  9,648   non‑malignant:   5,240   (cell_type='Malignant')
He2021             2,151 cells  ->  malignant:    827   non‑malignant:   1,324   (cell_type='Malignant')
Song2022          20,355 cells  ->  malignant:  4,193   non‑malignant:  16,162   (cell_type='Malignant')

TOTAL:  67,779 cells  ->  malignant: 28,953   non‑malignant:  38,826


/tmp/ipykernel_34004/3586130184.py:64: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adatas[0].concatenate(


Merged: 67,779 cells × 14,497 genes
Study names restored: ['Chen2021', 'Dong2020', 'He2021', 'Song2022']
Metadata retained: ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']

Cancer types: ['Prostate Cancer']
Malignant distribution:
is_malignant
0    38826
1    28953
Name: count, dtype: int64
Final clean dataset: 67,779 cells × 14,497 genes
Temporary files cleaned up.
